# Drone Fleet Optimization — Phase 1 Walkthrough

This notebook walks through the full Phase 1 pipeline end-to-end:

1. Generate a synthetic priority field
2. Produce boustrophedon strips
3. Assign strips with MILP — **makespan vs weighted objective**
4. Simulate both assignments
5. Inject failures and observe replanning
6. Add battery drain and see natural failures
7. Compare runs with metrics

All modules are decoupled: `field → optimizer → simulation → metrics → viz`.

In [ ]:
import sys
sys.path.insert(0, '..')

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

from src.field.generator import synthetic_field, generate_strips
from src.optimizer.milp import assign_strips, DroneSpec
from src.simulation.engine import simulate
from src.simulation.metrics import compute_metrics, compare_runs, plot_coverage_over_time, plot_battery_over_time, plot_metrics_bar
from src.viz.renderer import animate, CELL_CMAP

%matplotlib inline

## 1. Synthetic Field

Values in `[0, 1]` represent vegetation density / spray priority.
The gradient increases top-to-bottom (higher density crops toward one edge).

In [ ]:
NROWS, NCOLS = 8, 8

field_grid = synthetic_field(nrows=NROWS, ncols=NCOLS, seed=42)

fig, ax = plt.subplots(figsize=(5, 4))
im = ax.imshow(field_grid, cmap='YlGn', vmin=0, vmax=1)
plt.colorbar(im, ax=ax, label='Spray priority')
ax.set_title('Synthetic Field — Spray Priority')
ax.set_xlabel('Column'); ax.set_ylabel('Row')
plt.tight_layout()
plt.show()

print(f'Grid shape: {field_grid.shape}')
print(f'Priority range: {field_grid.min():.3f} – {field_grid.max():.3f}')

## 2. Strip Generation

Each row becomes one strip. Direction alternates per row (boustrophedon / lawnmower path)
so adjacent strips connect end-to-end with no wasted transit.

In [ ]:
strips = generate_strips(field_grid, seconds_per_cell=2.0)

print(f'{len(strips)} strips generated:\n')
for s in strips:
    bar = '█' * int(s.priority * 20)
    print(f'  Strip {s.id:2d} | priority={s.priority:.2f} {bar:<20} | time={s.time:.1f}s')

## 3. Fleet Setup

In [ ]:
N_DRONES = 3
drones = [DroneSpec(id=i, battery=100.0, spray_capacity=100.0) for i in range(N_DRONES)]

for d in drones:
    print(f'  Drone {d.id}: battery={d.battery}%, spray={d.spray_capacity}%')

## 4. MILP Optimization — Makespan vs Weighted

**Makespan mode**: minimize the maximum drone workload (balances the fleet).

**Weighted mode**: maximize priority-weighted coverage minus a makespan penalty.
The optimizer prefers to assign high-priority strips first. Useful when full coverage
is not guaranteed (e.g., limited battery).

In [ ]:
result_ms = assign_strips(strips, drones, objective_mode='makespan')
result_wt = assign_strips(strips, drones, objective_mode='weighted',
                          priority_weight=1.0, makespan_penalty=0.5)

for label, r in [('Makespan', result_ms), ('Weighted', result_wt)]:
    print(f'--- {label} ---')
    print(f'  Status: {r.status}  Makespan: {r.makespan}s  Solve time: {r.solve_time}s')
    for d_id, strip_ids in r.assignment.items():
        total_t = sum(s.time for s in strips if s.id in strip_ids)
        mean_p  = np.mean([s.priority for s in strips if s.id in strip_ids]) if strip_ids else 0
        print(f'  Drone {d_id}: strips={strip_ids}  workload={total_t:.1f}s  mean_priority={mean_p:.2f}')
    print()

## 5. Baseline Simulation — No Failures

In [ ]:
hist_baseline = simulate(
    strips=strips, drones=drones, result=result_ms,
    nrows=NROWS, ncols=NCOLS,
)
metrics_baseline = compute_metrics(hist_baseline, strips, NROWS, NCOLS)

print('Baseline (makespan, no failures):')
for k in ['coverage_pct', 'priority_coverage', 'makespan', 'replan_count', 'failed_drone_count', 'efficiency']:
    print(f'  {k:<25}: {metrics_baseline[k]}')

## 6. Failure Injection — Scripted Battery Failure

In [ ]:
failure_events = [
    {'timestep': 8,  'drone_id': 0, 'type': 'battery'},
    {'timestep': 20, 'drone_id': 1, 'type': 'comms', 'duration': 4},
]

hist_failures = simulate(
    strips=strips, drones=drones, result=result_ms,
    nrows=NROWS, ncols=NCOLS,
    failure_events=failure_events,
    replan_objective='makespan',
)
metrics_failures = compute_metrics(hist_failures, strips, NROWS, NCOLS)

print('With scripted failures:')
for k in ['coverage_pct', 'priority_coverage', 'makespan', 'replan_count', 'failed_drone_count']:
    print(f'  {k:<25}: {metrics_failures[k]}')

print('\nEvent log:')
for s in hist_failures:
    if s['event']:
        print(f'  t={s["timestep"]:3d}: {s["event"]}')

## 7. Battery Drain — Natural Failures

Instead of scripted events, drones now deplete battery as they fly.
Each drone starts at 100% and loses `battery_drain_per_cell` per cell traversed.

On this 8×8 grid, the MILP assigns ~24 cells to two drones and ~16 to one.
With drain=5.5%, a 24-cell drone fails at cell ~18 — mid-task.
The replanner redistributes remaining work to the other drones,
but they too are running low, causing a **cascading failure**.
This is the core failure mode this project is designed to study.

Try `DRAIN = 2.0` (all survive), `5.5` (cascade), `8.0` (fast collapse).

In [ ]:
DRAIN = 5.5   # % battery per cell — try 2.0 (all survive), 5.5 (cascade), 8.0 (fast collapse)

hist_drain = simulate(
    strips=strips, drones=drones, result=result_ms,
    nrows=NROWS, ncols=NCOLS,
    battery_drain_per_cell=DRAIN,
    replan_objective='makespan',
)
metrics_drain = compute_metrics(hist_drain, strips, NROWS, NCOLS)

print(f'Battery drain ({DRAIN}%/cell):')
for k in ['coverage_pct', 'priority_coverage', 'makespan', 'replan_count', 'failed_drone_count']:
    print(f'  {k:<25}: {metrics_drain[k]}')

print('\nEvent log:')
for s in hist_drain:
    if s['event']:
        print(f'  t={s["timestep"]:3d}: {s["event"]}')

## 8. Weighted Objective + Battery Drain

When battery is scarce, the **weighted objective** tries to maximise priority-weighted
coverage rather than just balance load. Ideally, high-priority strips are covered
before drones run out.

On a small grid the two objectives may produce identical assignments (the MILP
solution is often unique when constraints are tight). The real divergence appears
when the MILP is given explicit battery capacity limits — a natural Phase 2 extension:
add `battery_capacity_seconds` to `assign_strips()` so the optimizer plans around
the range limit rather than discovering it mid-simulation.

In [ ]:
hist_drain_wt = simulate(
    strips=strips, drones=drones, result=result_wt,
    nrows=NROWS, ncols=NCOLS,
    battery_drain_per_cell=DRAIN,
    replan_objective='weighted',
    replan_priority_weight=1.0,
    replan_makespan_penalty=0.5,
)
metrics_drain_wt = compute_metrics(hist_drain_wt, strips, NROWS, NCOLS)

print(f'Weighted + battery drain ({DRAIN}%/cell):')
for k in ['coverage_pct', 'priority_coverage', 'makespan', 'replan_count', 'failed_drone_count']:
    print(f'  {k:<25}: {metrics_drain_wt[k]}')

## 9. Metrics Comparison

In [ ]:
all_runs = {
    'Baseline':         metrics_baseline,
    'Scripted failures': metrics_failures,
    f'Drain {DRAIN}%/cell': metrics_drain,
    f'Drain+Weighted':  metrics_drain_wt,
}

comparison = compare_runs(all_runs)

print(f'{"Metric":<25}', '  '.join(f'{k:<18}' for k in all_runs))
print('-' * 95)
for metric, vals in comparison.items():
    row = '  '.join(f'{v!s:<18}' for v in vals.values())
    print(f'{metric:<25}{row}')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

# Coverage over time
plot_coverage_over_time(
    runs={
        'Baseline':          metrics_baseline['cells_per_step'],
        'Scripted failures': metrics_failures['cells_per_step'],
        f'Drain {DRAIN}%':   metrics_drain['cells_per_step'],
        'Drain+Weighted':    metrics_drain_wt['cells_per_step'],
    },
    total_cells=NROWS * NCOLS,
    title='Coverage over time',
    ax=axes[0],
)

# Battery over time for the drain run
plot_battery_over_time(
    battery_series=metrics_drain['battery_series'],
    replan_steps=metrics_drain['replan_steps'],
    title=f'Battery over time (drain={DRAIN}%/cell)',
    ax=axes[1],
)

plt.tight_layout()
plt.show()

## 10. Animation

Render the battery-drain run. Change `state_history` to any of the `hist_*` variables above.

In [ ]:
from matplotlib import rc
rc('animation', html='jshtml')   # renders inline in Jupyter

anim = animate(
    state_history=hist_drain,
    nrows=NROWS,
    ncols=NCOLS,
    interval_ms=150,
    show=False,   # don't open a separate window in Jupyter
)
anim   # display inline

In [ ]:
# Save a GIF for the README
import os
os.makedirs('../results', exist_ok=True)

animate(
    state_history=hist_drain,
    nrows=NROWS,
    ncols=NCOLS,
    interval_ms=150,
    save_path='../results/phase1_demo.gif',
    show=False,
)
print('Saved: results/phase1_demo.gif')